# Phase 2 — Prototype d'entraînement

Notebook de prototypage avant `scripts/train.py`. Les fonctions définies ici
(`set_seed`, `build_model`, `predict`) seront déplacées dans `src/` avec leurs tests.

Contenu :
1. configuration et chargement des partitions (`train_val.csv`, jamais le test) ;
2. `set_seed`, `build_model`, `predict` ;
3. **test de bon sens 1** : perte initiale ≈ ln(7) ;
4. **test de bon sens 2** : sur-apprendre 32 images ;
5. une époque sur le fold 0 avec sauvegarde des logits.

Fonctionne en local (CPU) et sur Kaggle (GPU) : le dépôt est détecté automatiquement.

In [ ]:
import os, sys, math, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import models

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.classes import CLASSES, NUM_CLASSES
from src.dataset import DermDataset
from src.train import train_one_epoch
from src.transforms import get_train_transforms, get_val_transforms
from src.utils import compute_metrics, get_class_weights

QUICK = bool(os.environ.get("QUICK"))      # exécution rapide pour vérifier le notebook
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_DIR = ROOT / "data/interim/256"
if not IMAGE_DIR.is_dir():
    IMAGE_DIR = ROOT / "data/raw"
VAL_FOLD, SEED, BATCH_SIZE = 0, 0, 64
NUM_WORKERS = 0 if QUICK else 2
print(f"device={DEVICE} | images={IMAGE_DIR} | QUICK={QUICK}")

In [ ]:
# Les partitions : on ne lit que train_val.csv, le test reste intouché.
train_val = pd.read_csv(ROOT / "data/processed/train_val.csv")
assert (train_val["split"] == "train_val").all()
assert (train_val["label"] == train_val["dx"].map(CLASSES.index)).all()

train_df = train_val[train_val["fold"] != VAL_FOLD].reset_index(drop=True)
val_df = train_val[train_val["fold"] == VAL_FOLD].reset_index(drop=True)
assert set(train_df["lesion_id"]).isdisjoint(val_df["lesion_id"])
print(len(train_df), "images d'entraînement |", len(val_df), "images de validation")

## `set_seed`
Graine pour `random`, `numpy`, `torch`, plus un `Generator` et un `worker_init_fn` pour le `DataLoader`.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(df, transform, *, shuffle, generator=None):
    dataset = DermDataset(df, IMAGE_DIR, transform=transform)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
                      generator=generator)

## `build_model`
`trainable` : `"head"` (seule `fc` apprend), `"layer4"` (`layer4` + `fc`) ou `"all"`.

In [ ]:
def build_model(name="resnet18", pretrained=True, trainable="head"):
    assert name == "resnet18"
    weights = None
    if pretrained:
        try:
            weights = models.ResNet18_Weights.IMAGENET1K_V1
            model = models.resnet18(weights=weights)
        except Exception as error:  # pas d'internet : poids ImageNet indisponibles
            print("Poids ImageNet indisponibles, modèle aléatoire :", type(error).__name__)
            model = models.resnet18(weights=None)
    else:
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(512, NUM_CLASSES)

    for param in model.parameters():
        param.requires_grad = trainable == "all"
    if trainable == "layer4":
        for param in model.layer4.parameters():
            param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    return model


def trainable_names(model):
    return sorted({n.split(".")[0] for n, p in model.named_parameters() if p.requires_grad})


for mode in ("head", "layer4", "all"):
    print(mode, "->", trainable_names(build_model(pretrained=False, trainable=mode)))

## `predict` et `evaluate`
`predict` renvoie `(logits, labels)` dans l'ordre du dataset ; `evaluate` ajoute les métriques.

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        all_logits.append(model(images.to(device)).cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


def evaluate(model, loader, criterion, device):
    logits, labels = predict(model, loader, device)
    metrics = compute_metrics(labels.numpy(), logits.argmax(dim=1).numpy())
    metrics["loss"] = criterion(logits, labels).item()
    return metrics, logits, labels

## Test de bon sens 1 — perte initiale

Avant tout entraînement, la perte doit valoir environ ln(7) ≈ 1,95.

*Pourquoi :* si le modèle ne sait encore rien, il répartit sa confiance à peu près
uniformément sur les 7 classes ; la cross-entropy d'une loi uniforme vaut
−ln(1/7) = ln(7). Une valeur très supérieure signale des logits trop grands
(mauvaise initialisation, normalisation absente) ; très inférieure, une fuite de labels.

*Attention :* la mesure doit se faire sur des images **mélangées**. Un lot pris dans
`val_df` (trié par lésion) est presque mono-classe : la perte dépend alors de la classe
que la tête initiale favorise par hasard, et peut s'éloigner de ln(7). Avec une tête
`fc` initialisée aléatoirement sur des caractéristiques ImageNet, la valeur moyenne est
proche de ln(7), avec un léger excès dû à la dispersion des logits.

In [ ]:
generator = set_seed(SEED)
val_loader = make_loader(val_df, get_val_transforms(), shuffle=False)
model = build_model(pretrained=True, trainable="head").to(DEVICE)
criterion = nn.CrossEntropyLoss()

sample_df = val_df.sample(64 if QUICK else 512, random_state=SEED).reset_index(drop=True)
sample_loader = make_loader(sample_df, get_val_transforms(), shuffle=False)
logits, labels = predict(model, sample_loader, DEVICE)
initial_loss = criterion(logits, labels).item()
print(f"perte initiale = {initial_loss:.3f} | ln(7) = {math.log(7):.3f} "
      f"| sur {len(sample_df)} images mélangées")

## Test de bon sens 2 — sur-apprendre 32 images

`trainable="all"`, sans augmentation, 32 images fixes : la perte d'entraînement doit
tendre vers 0 en une centaine d'itérations. Sinon le pipeline a un bug (labels décalés,
normalisation, learning rate).

In [ ]:
generator = set_seed(SEED)
small_df = train_df.sample(32, random_state=SEED).reset_index(drop=True)
small_loader = make_loader(small_df, get_val_transforms(), shuffle=False)
images, labels = next(iter(small_loader))
images, labels = images.to(DEVICE), labels.to(DEVICE)

model = build_model(pretrained=True, trainable="all").to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

n_iter = 10 if QUICK else 100
model.train()
losses = []
for it in range(n_iter):
    optimizer.zero_grad()
    loss = criterion(model(images), labels)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if it % 10 == 0 or it == n_iter - 1:
        print(f"it {it:3d} | perte {loss.item():.4f}")

if not QUICK:
    assert losses[-1] < 0.05, "le pipeline n'arrive pas à sur-apprendre 32 images"
    print("OK : sur-apprentissage réussi")

## Une époque sur le fold 0 (tête seule) avec sauvegarde des logits

Entraînement sur les folds 1 à 4, validation sur le fold 0. Les logits (et non les
probabilités) sont sauvegardés : ils permettent plus tard le *temperature scaling*
(on divise les logits par T) et l'abstention, sans réentraîner.

In [ ]:
generator = set_seed(SEED)
train_loader = make_loader(train_df, get_train_transforms(), shuffle=True, generator=generator)
val_loader = make_loader(val_df, get_val_transforms(), shuffle=False)

model = build_model(pretrained=True, trainable="head").to(DEVICE)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

start = time.perf_counter()
train_loss = train_one_epoch(model, train_loader, optimizer=optimizer,
                             criterion=criterion, device=DEVICE) if not QUICK else float("nan")
duration = time.perf_counter() - start
print(f"perte train = {train_loss:.4f} | durée = {duration:.1f} s")

In [ ]:
if QUICK:
    val_df_eval = val_df.head(128)
    val_loader = make_loader(val_df_eval, get_val_transforms(), shuffle=False)
else:
    val_df_eval = val_df

metrics, logits, labels = evaluate(model, val_loader, criterion, DEVICE)
print({k: round(v, 4) for k, v in metrics.items()})

out = pd.DataFrame(logits.numpy(), columns=[f"logit_{i}" for i in range(NUM_CLASSES)])
out.insert(0, "label", labels.numpy())
out.insert(0, "lesion_id", val_df_eval["lesion_id"].values)
out.insert(0, "image_id", val_df_eval["image_id"].values)
assert out["image_id"].tolist() == val_df_eval["image_id"].tolist()   # ordre du dataset
assert logits.shape == (len(val_df_eval), NUM_CLASSES)
out.head()

## Suite
- Déplacer `set_seed`, `build_model`, `predict` dans `src/` et ajouter leurs tests.
- Écrire `scripts/train.py` (config YAML, arrêt précoce sur la macro-F1 de validation,
  écriture de `config.yaml`, `history.csv`, `best.pt`, `val_logits.csv`, `metrics.json`).
- Vérifier la reproductibilité : deux lancements identiques, même macro-F1 à 1e-4 près.